# Lesson 4A: Temporal Difference Learning - Theory

## Introduction

**Temporal Difference (TD) Learning** is the most important algorithm in modern RL.

It combines the best of DP and MC:
- **From DP**: Bootstrap—update using estimates of future values (no need for full episodes)
- **From MC**: Sample experience—don't need to know environment model

### Why TD is Revolutionary

**DP**: Fast but needs full model
**MC**: Model-free but slow, high variance, needs complete episodes
**TD**: Model-free, low variance, learns online (before episode ends)

### Learning Objectives

1. Understand **bootstrapping** and TD error
2. Derive **TD(0)** prediction algorithm
3. Learn **Sarsa** (on-policy) control
4. Learn **Q-learning** (off-policy) control
5. Understand bias-variance tradeoff
6. Convergence guarantees

## Table of Contents

1. [Setup](#setup)
2. [Bootstrap & TD Error](#bootstrap)
3. [TD(0) Prediction](#td-prediction)
4. [Bias-Variance Tradeoff](#bias-variance)
5. [Sarsa: On-Policy Control](#sarsa)
6. [Q-Learning: Off-Policy Control](#qlearning)
7. [Expected Sarsa](#expected-sarsa)
8. [Convergence Guarantees](#convergence)
9. [Algorithm Comparison](#comparison)
10. [Summary](#summary)

<a name="setup"></a>
## Setup

In [ ]:
import subprocess, sys
packages = ['numpy', 'matplotlib', 'seaborn', 'pandas']
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Ready for TD Learning theory')

<a name="bootstrap"></a>
## Bootstrap & TD Error

### The Core Idea

Instead of waiting for full return like MC:
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ...$$

TD uses **bootstrap**—estimate future using current value estimate:
$$\hat{G}_t = R_{t+1} + \gamma V(S_{t+1})$$

### Why Bootstrap?

1. **Online learning**: Update immediately after each step (no wait for episode end)
2. **Lower variance**: Use one sample (R_t+1) + estimate (V), not full trajectory
3. **Real-time**: Act before episode finishes

### TD Error

$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

Measures: (observed reward + estimated future) - (current estimate)

Positive: Underestimated state value → increase it
Negative: Overestimated state value → decrease it

<a name="td-prediction"></a>
## TD(0) Prediction

### Algorithm

TD(0) estimates V^π for a given policy π:

```
Initialize V(s) for all s

loop:
  Generate experience: S_t, A_t ~ π, R_{t+1}, S_{t+1}
  TD error: δ ← R_{t+1} + γV(S_{t+1}) - V(S_t)
  Update: V(S_t) ← V(S_t) + α·δ
  S_t ← S_{t+1}
```

### Key Properties

1. **Unbiased**: E[R_{t+1} + γV(S_{t+1})] = E[V^π(S_t)] if V converged
2. **Bootstrap**: Uses V(S_{t+1}) estimate
3. **Online**: Updates after each step
4. **Model-free**: Only needs samples, not P or R
5. **Convergence**: To V^π with probability 1 (under conditions)

### Convergence Conditions

V converges to V^π if:
1. All states visited infinitely often
2. Learning rate α decays: Σα = ∞, Σα² < ∞
3. Sufficiently long learning

<a name="bias-variance"></a>
## Bias-Variance Tradeoff

### Three Estimators Compared

**MC**: G_t = R_{t+1} + γR_{t+2} + γ²R_{t+3} + ...
- Unbiased
- High variance (depends on all future rewards)

**DP**: R_{t+1} + γV(S_{t+1}) (if V=V^π)
- No variance (deterministic)
- Biased (depends on accuracy of V)

**TD**: R_{t+1} + γV(S_{t+1}) (with learned V)
- Some bias (V may be inaccurate)
- Lower variance than MC (one sample + estimate)
- **Sweet spot**: Bias+variance < MC or DP alone

### Empirical Insight

TD typically learns faster than MC because:
- MC: High variance → needs many samples to average out
- TD: Lower variance → needs fewer samples to converge

In [ ]:
# Visualize bias-variance tradeoff
fig, ax = plt.subplots(figsize=(10, 6))

alpha = np.linspace(0, 1, 100)  # α=0 is DP, α=1 is MC
bias_sq = alpha**2 * 0.1  # DP bias^2 = 0, MC bias^2 = 0.1
variance = (1 - alpha)**2 * 1.0  # DP variance = 0, MC variance = 1.0
total_error = bias_sq + variance

ax.plot(alpha, bias_sq, label='Bias²', linewidth=2.5)
ax.plot(alpha, variance, label='Variance', linewidth=2.5)
ax.plot(alpha, total_error, label='Total Error', linewidth=3, linestyle='--')

ax.fill_between(alpha, bias_sq, bias_sq + variance, alpha=0.1, color='blue')
ax.axvline(x=0.3, color='red', linestyle=':', linewidth=2, label='TD sweet spot')
ax.axhline(y=np.min(total_error), color='green', linestyle=':', alpha=0.5, linewidth=1)

ax.set_xlabel('Bootstrap Amount (0=DP, 1=MC)', fontsize=11)
ax.set_ylabel('Error', fontsize=11)
ax.set_title('Bias-Variance Tradeoff: DP vs MC vs TD', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

ax.text(0.05, 0.95, 'DP\n(low variance,\nhigh bias)', fontsize=9, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8), verticalalignment='top')
ax.text(0.85, 0.95, 'MC\n(high variance,\nlow bias)', fontsize=9, transform=ax.transAxes,
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8), verticalalignment='top', horizontalalignment='right')

plt.tight_layout()
plt.show()

<a name="sarsa"></a>
## Sarsa: On-Policy Control

### Algorithm Name

**Sarsa** = State-Action-Reward-State-Action (the tuple used in update)

### Update Rule

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

Bootstrap using next action A_{t+1} (the one we're about to take).

### On-Policy

Learn the policy you're following (e.g., ε-greedy).
- Converges to ε-greedy optimal (not fully optimal π*)
- Safe: only learns from safe behavior
- Slower: ε exploration adds suboptimality

### Convergence

Q → Q^π with probability 1 if:
1. ε-greedy exploration (or Boltzmann)
2. Learning rates decay properly
3. All (s,a) visited infinitely often

<a name="qlearning"></a>
## Q-Learning: Off-Policy Control

### Algorithm

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)]$$

Bootstrap using **best possible next action** max_a Q(S_{t+1}, a), regardless of actual A_{t+1}.

### Off-Policy

Learn optimal π* while following exploratory behavior μ.
- More aggressive: assumes best possible future
- Faster convergence: less reliant on safe exploration
- Can overestimate: positive bias if Q overestimates

### Convergence

Q → Q* (optimal) with probability 1 under:
1. All (s,a) visited infinitely often (exploration)
2. Learning rates decay
3. No model needed

### Key Advantage

Learns optimal policy from suboptimal experience—critical for real-world learning.

<a name="expected-sarsa"></a>
## Expected Sarsa

### Balance Between Sarsa and Q-Learning

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \mathbb{E}[Q(S_{t+1}, A_{t+1})] - Q(S_t, A_t)]$$

Use expected value under current policy instead of:
- max (too aggressive, overestimates like Q-learning)
- next actual action (underestimates like Sarsa)

### Benefits

1. **Lower variance**: Expected value smoother than sampling
2. **Reduced overestimation**: Conservative vs Q-learning
3. **Stable learning**: Better for off-policy
4. **Off-policy capable**: Like Q-learning but more stable

<a name="convergence"></a>
## Convergence Guarantees

### Convergence Conditions

All TD algorithms converge to optimal Q* or V^π if:

1. **Exploration**: All (s,a) visited infinitely often
2. **Learning rate decay**: ∑α(t) = ∞, ∑α(t)² < ∞
3. **MDP structure**: Finite S, A, and rewards bounded

### Rate of Convergence

- **Slower than DP**: No access to true model
- **Faster than MC**: Lower variance from bootstrapping
- **Practical speed**: Often fastest in real-world problems

### Theoretical Result

Under conditions above, TD(0) for Q-learning:
- Converges to Q* with probability 1
- Error decreases at rate O(1/√t) (optimal for stochastic)
- Sample efficient compared to MC

<a name="comparison"></a>
## Algorithm Comparison

| Aspect | MC | TD/Sarsa | TD/Q-Learning | DP |
|--------|----|---------|-----------|----- |
| **Bootstraps?** | No | Yes | Yes | Yes |
| **Needs model?** | No | No | No | Yes |
| **Sample efficient** | Low | High | High | N/A |
| **Variance** | High | Medium | Medium-High* | None |
| **Bias** | None | Some | Some | Depends on V |
| **Converges to** | V^π, π* | π_ε | Q* | Q*, π* |
| **Online** | No | Yes | Yes | N/A |
| **Off-policy** | Possible (IS) | Hard | Natural | N/A |
| **Convergence proof** | Yes | Yes | Yes | Yes |

\*Q-learning can overestimate, inflating variance

### Modern Consensus

**TD methods are the gold standard**:
- Model-free (real-world)
- Online (streaming data)
- Sample efficient
- Foundation for deep RL

<a name="summary"></a>
## Summary

### Key Concepts

1. **Bootstrap**: Use value estimates for future (not just samples)
2. **TD Error**: R + γV(s') - V(s)
3. **TD(0)**: One-step bootstrap for prediction
4. **Sarsa**: On-policy Q-learning using actual next action
5. **Q-Learning**: Off-policy using best next action
6. **Expected Sarsa**: Middle ground with expected value
7. **Bias-Variance**: TD trades off DP and MC

### Algorithm Taxonomy

```
RL Algorithms
├── DP: Model + full lookahead (fast but needs model)
├── MC: Samples + full episodes (model-free but high variance)
└── TD: Samples + bootstrapped estimates (model-free + efficient)
    ├── TD(0): One-step prediction
    ├── Sarsa: On-policy control
    ├── Q-Learning: Off-policy control ⭐
    └── Expected Sarsa: Stable off-policy
```

### Why Q-Learning Dominates

- Learns optimal policy from any behavior
- Sample efficient (low variance via bootstrap)
- No model needed
- Simple implementation
- Foundation for Deep Q-Networks (Lesson 7)

### Next: Lesson 4B (Practical)

Implement Sarsa and Q-learning on CliffWalking—see the off-policy advantage live.